In [ ]:
CPP_SOURCE = r"""// Identify a molecule by finding the training spectra that look like it.
//
// The shape of the problem, measured before any of this was written: 400 test
// molecules, 1,213 spectra, all from one instrument; 2.54 million training
// spectra over 276 thousand molecules. Every test precursor mass exists in the
// training set, but within 10 ppm the median mass has 105 distinct molecules
// behind it and the worst has 558 - so the mass narrows the field to about a
// hundred and the spectrum has to do the rest. Only 38 of the 400 have few
// enough candidates that mass alone would fill the answer.
//
// That is a billion peak-list comparisons, which is why this is not Python.
//
// Build:
//   g++ -O3 -std=c++17 -pthread -o match match.cpp
//
// Usage:
//   match <dir> <ppm> <peak_tol> <topk> <lib_stem> <qry_stem> [sigma_ppm] [alpha]

#include <algorithm>
#include <atomic>
#include <cmath>
#include <cstdint>
#include <cstdio>
#include <cstring>
#include <fstream>
#include <string>
#include <thread>
#include <unordered_map>
#include <vector>

namespace {

struct Head {            // one per spectrum, as written by export_bin.py
  float precursor;
  int32_t mol;
  int32_t off;           // index into the peak arrays
  int32_t n;             // how many peaks
  int32_t tag;           // adduct * 16 + instrument, -1 where unknown
};

struct Store {
  std::vector<Head> head;
  std::vector<float> mz, inten;
  std::vector<std::string> name, smiles;
  std::vector<float> ent;        // Shannon entropy of each spectrum, after weighting

  const float* mzs(const Head& h) const { return mz.data() + h.off; }
  const float* its(const Head& h) const { return inten.data() + h.off; }
  int adduct(const Head& h) const { return h.tag < 0 ? -1 : h.tag / 16; }
  int instr(const Head& h) const { return h.tag < 0 ? -1 : h.tag % 16; }
};

std::vector<char> slurp(const std::string& path) {
  std::ifstream f(path, std::ios::binary | std::ios::ate);
  if (!f) { std::fprintf(stderr, "cannot read %s\n", path.c_str()); std::exit(1); }
  std::streamsize n = f.tellg();
  f.seekg(0);
  std::vector<char> buf((size_t(n)));
  f.read(buf.data(), n);
  return buf;
}

Store load(const std::string& dir, const std::string& stem) {
  Store s;
  std::vector<char> hb = slurp(dir + "/" + stem + "_head.bin");
  s.head.resize(hb.size() / sizeof(Head));
  std::memcpy(s.head.data(), hb.data(), s.head.size() * sizeof(Head));

  std::vector<char> pb = slurp(dir + "/" + stem + "_peaks.bin");
  // Each spectrum wrote its m/z block then its intensity block, so the pairs
  // are interleaved per spectrum rather than globally. Split them out once.
  size_t total = 0;
  for (const Head& h : s.head) total += size_t(h.n);
  s.mz.resize(total);
  s.inten.resize(total);
  const float* p = reinterpret_cast<const float*>(pb.data());
  size_t cur = 0;
  for (const Head& h : s.head) {
    std::memcpy(s.mz.data() + cur, p, size_t(h.n) * sizeof(float));
    p += h.n;
    std::memcpy(s.inten.data() + cur, p, size_t(h.n) * sizeof(float));
    p += h.n;
    cur += size_t(h.n);
  }

  std::ifstream f(dir + "/" + stem + "_mols.txt");
  std::string line;
  while (std::getline(f, line)) {
    if (!line.empty() && line.back() == '\r') line.pop_back();
    size_t t = line.find('\t');
    s.name.push_back(line.substr(0, t));
    s.smiles.push_back(t == std::string::npos ? "" : line.substr(t + 1));
  }
  // Entropy weighting, Li et al. 2021.
  //
  // A plain dot product treats a three-peak spectrum and an eighty-peak one as
  // equally trustworthy. They are not: a low-entropy spectrum carries few
  // independent constraints and matches far too many things by accident. The
  // fix is to flatten the confident-looking ones before comparing, raising the
  // intensities to w = 0.25 + 0.25 S when the entropy S is below 3, and to
  // score with entropy similarity instead of cosine. The one published
  // measurement on this competition puts Class-1 MRR at 0.919 with it against
  // 0.895 with sqrt intensities.
  s.ent.resize(s.head.size());
  for (size_t i = 0; i < s.head.size(); ++i) {
    const Head& h = s.head[i];
    float* it = s.inten.data() + h.off;
    double sum = 0.0;
    for (int k = 0; k < h.n; ++k) sum += double(it[k]);
    if (sum <= 0.0) { s.ent[i] = 0.0f; continue; }
    for (int k = 0; k < h.n; ++k) it[k] = float(double(it[k]) / sum);
    double S = 0.0;
    for (int k = 0; k < h.n; ++k)
      if (it[k] > 0.0f) S -= double(it[k]) * std::log(double(it[k]));
    if (S < 3.0) {
      double w = 0.25 + 0.25 * S, s2 = 0.0;
      for (int k = 0; k < h.n; ++k) { it[k] = float(std::pow(double(it[k]), w)); s2 += it[k]; }
      if (s2 > 0.0) {
        S = 0.0;
        for (int k = 0; k < h.n; ++k) {
          it[k] = float(double(it[k]) / s2);
          if (it[k] > 0.0f) S -= double(it[k]) * std::log(double(it[k]));
        }
      }
    }
    s.ent[i] = float(S);
  }

  std::printf("%s: %zu spectra, %zu molecules, %zu peaks\n",
              stem.c_str(), s.head.size(), s.name.size(), total);
  return s;
}

// Cosine over matched peaks, the standard way these libraries are searched.
//
// Peaks match when their m/z agree within a tolerance; both lists are sorted,
// so one walk of each is enough. Intensities are square-rooted first, which is
// what every spectral library search does - it stops one tall peak from
// deciding the whole score, and tall peaks are the least informative part of a
// fragmentation pattern.
float cosine(const float* amz, const float* ait, int an,
             const float* bmz, const float* bit, int bn, float tol) {
  double dot = 0.0, na = 0.0, nb = 0.0;
  for (int i = 0; i < an; ++i) na += double(ait[i]);
  for (int j = 0; j < bn; ++j) nb += double(bit[j]);
  if (na <= 0.0 || nb <= 0.0) return 0.0;

  int j = 0;
  for (int i = 0; i < an; ++i) {
    while (j < bn && bmz[j] < amz[i] - tol) ++j;
    int k = j;
    float best = 0.0f;
    while (k < bn && bmz[k] <= amz[i] + tol) {
      best = std::max(best, bit[k]);
      ++k;
    }
    if (best > 0.0f) dot += std::sqrt(double(ait[i])) * std::sqrt(double(best));
  }
  double norma = 0.0, normb = 0.0;
  for (int i = 0; i < an; ++i) norma += double(ait[i]);
  for (int j2 = 0; j2 < bn; ++j2) normb += double(bit[j2]);
  return float(dot / std::sqrt(norma * normb + 1e-12));
}

// Entropy similarity. Both spectra are already normalised to sum 1 and
// weighted, so their merge is the average of the two and
//     sim = 1 - (2 S_AB - S_A - S_B) / ln 4
// which is 1 for identical spectra and 0 for disjoint ones.
float entropy_sim(const float* amz, const float* ait, int an, float sa,
                  const float* bmz, const float* bit, int bn, float sb,
                  float tol) {
  double sab = 0.0;
  int i = 0, j = 0;
  auto add = [&sab](double q) { if (q > 0.0) sab -= q * std::log(q); };
  while (i < an && j < bn) {
    if (bmz[j] < amz[i] - tol) { add(0.5 * double(bit[j])); ++j; }
    else if (bmz[j] > amz[i] + tol) { add(0.5 * double(ait[i])); ++i; }
    else { add(0.5 * (double(ait[i]) + double(bit[j]))); ++i; ++j; }
  }
  for (; i < an; ++i) add(0.5 * double(ait[i]));
  for (; j < bn; ++j) add(0.5 * double(bit[j]));
  double v = 1.0 - (2.0 * sab - double(sa) - double(sb)) / std::log(4.0);
  return float(v < 0.0 ? 0.0 : v > 1.0 ? 1.0 : v);
}

// A molecule's several query spectra merged into one before matching, instead
// of matching each separately and taking the best candidate score across
// them. Close peaks (within tol) are summed rather than kept apart, then the
// merge is renormalised and entropy-weighted exactly as load() treats a
// stored spectrum - a merged spectrum is just a spectrum.
//
// The tutorial notebook reports feeding the wrong one of these two costs
// ~0.02 MRR; which one is wrong is exactly what this flag is for measuring.
struct MergedSpec { std::vector<float> mz, it; float ent = 0.0f; };

MergedSpec merge_query(const Store& te, const std::vector<int>& sis, float tol) {
  std::vector<std::pair<float, float> > peaks;
  for (int si : sis) {
    const Head& q = te.head[si];
    const float* mz = te.mzs(q);
    const float* it = te.its(q);
    for (int k = 0; k < q.n; ++k) peaks.emplace_back(mz[k], it[k]);
  }
  std::sort(peaks.begin(), peaks.end());
  MergedSpec out;
  for (auto& pr : peaks) {
    if (!out.mz.empty() && pr.first - out.mz.back() <= tol) out.it.back() += pr.second;
    else { out.mz.push_back(pr.first); out.it.push_back(pr.second); }
  }
  double sum = 0.0;
  for (float x : out.it) sum += double(x);
  if (sum <= 0.0) return out;
  for (float& x : out.it) x = float(double(x) / sum);
  double S = 0.0;
  for (float x : out.it) if (x > 0.0f) S -= double(x) * std::log(double(x));
  if (S < 3.0) {
    double w = 0.25 + 0.25 * S, s2 = 0.0;
    for (float& x : out.it) { x = float(std::pow(double(x), w)); s2 += double(x); }
    if (s2 > 0.0) {
      S = 0.0;
      for (float& x : out.it) {
        x = float(double(x) / s2);
        if (x > 0.0f) S -= double(x) * std::log(double(x));
      }
    }
  }
  out.ent = float(S);
  return out;
}

}  // namespace

int main(int argc, char** argv) {
  std::string dir = argc > 1 ? argv[1] : ".";
  float ppm = argc > 2 ? float(std::atof(argv[2])) : 10.0f;
  float tol = argc > 3 ? float(std::atof(argv[3])) : 0.01f;
  int topk = argc > 4 ? std::atoi(argv[4]) : 25;
  // Swap in the holdout by name, so the thing that is measured is the thing
  // that is submitted and not a second implementation of it.
  std::string lib_stem = argc > 5 ? argv[5] : "train";
  std::string qry_stem = argc > 6 ? argv[6] : "test";
  // How far the precursor is allowed to be wrong, and how much being wrong
  // costs. See the scoring loop.
  float sigma = argc > 7 ? float(std::atof(argv[7])) : 3.0f;
  float alpha = argc > 8 ? float(std::atof(argv[8])) : 0.1f;
  int simmode = argc > 9 ? std::atoi(argv[9]) : 1;   // 1 entropy, 0 sqrt cosine
  // 0 off, 1 soft bonus for a same-instrument hit, 2 hard-require same
  // instrument when both are known. Cross-instrument fragmentation patterns
  // differ systematically (declustering voltage, collision cell physics), so
  // this is worth a controlled measurement rather than an assumption either
  // way: a hard filter can delete the only reference a Class-1 molecule has.
  int instr_mode = argc > 10 ? std::atoi(argv[10]) : 0;
  float instr_bonus = argc > 11 ? float(std::atof(argv[11])) : 0.1f;
  // Match each of a molecule's spectra separately and keep the best candidate
  // score across them (0), or merge the spectra into one before matching (1).
  int merge_mode = argc > 12 ? std::atoi(argv[12]) : 0;

  Store tr = load(dir, lib_stem);
  Store te = load(dir, qry_stem);

  // Training spectra sorted by precursor mass, so a candidate window is a
  // binary search rather than a scan of two and a half million rows.
  std::vector<int> order(tr.head.size());
  for (size_t i = 0; i < order.size(); ++i) order[i] = int(i);
  std::sort(order.begin(), order.end(),
            [&](int a, int b) { return tr.head[a].precursor < tr.head[b].precursor; });
  std::vector<float> keys(order.size());
  for (size_t i = 0; i < order.size(); ++i) keys[i] = tr.head[order[i]].precursor;

  // Group the test spectra by molecule: several spectra of one molecule are
  // evidence about the same answer and their scores add.
  std::unordered_map<int, std::vector<int> > by_mol;
  for (size_t i = 0; i < te.head.size(); ++i) by_mol[te.head[i].mol].push_back(int(i));

  std::vector<int> mol_ids;
  for (const auto& kv : by_mol) mol_ids.push_back(kv.first);
  std::sort(mol_ids.begin(), mol_ids.end());

  std::vector<std::string> answer(mol_ids.size());
  std::vector<int> hit_rank(mol_ids.size(), -1);
  // How many candidates the TIGHTEST pass alone found, before any widening -
  // a separate signal from "25 candidates exist", which the staged widening
  // guarantees almost by construction and so cannot tell a confident Class-1
  // hit apart from 25 slots padded out with whatever the loosest pass found.
  std::vector<int> tight_count(mol_ids.size(), 0);
  unsigned nthread = std::max(1u, std::thread::hardware_concurrency());
  std::vector<std::thread> pool;
  std::atomic<size_t> next(0);

  for (unsigned t = 0; t < nthread; ++t) {
    pool.emplace_back([&]() {
      std::unordered_map<int, float> best;      // train molecule -> best score
      for (;;) {
        size_t idx = next++;
        if (idx >= mol_ids.size()) break;
        best.clear();

        MergedSpec merged;
        if (merge_mode) merged = merge_query(te, by_mol[mol_ids[idx]], tol);

        // One window, and being near the middle of it is worth something.
        //
        // The window used to start at 0.05 ppm and widen by 3x until 25
        // candidates existed, with a flat penalty per widening. That was tuned
        // on a holdout carved out of train, where it looked twenty times
        // better than a 1 ppm window - because train's precursor_mz is
        // calculated rather than measured (63.5% of (molecule, adduct) groups
        // are bit-identical, p99 spread under 1 ppm), so a query matched its
        // own library rows to the last digit. Re-run against queries carrying
        // 3 ppm of instrument error, which is what a real test spectrum has,
        // and the staged scheme drops from 0.481 MRR to 0.158.
        //
        // So the mass is evidence, not a gate: a candidate 1 ppm away is
        // better than one 8 ppm away, and the cost should grow smoothly rather
        // than in steps of a whole point at arbitrary 3x boundaries.
        // Gaussian in the ppm error, which is what the error actually is.
        float ppm_used = ppm;
        for (int pass = 0; pass < 10 && int(best.size()) < topk; ++pass) {
          for (int si : by_mol[mol_ids[idx]]) {
            const Head& q = te.head[si];
            float window = q.precursor * ppm_used * 1e-6f;
            size_t lo = std::lower_bound(keys.begin(), keys.end(),
                                         q.precursor - window) - keys.begin();
            size_t hi = std::upper_bound(keys.begin(), keys.end(),
                                         q.precursor + window) - keys.begin();
            for (size_t p = lo; p < hi; ++p) {
              const Head& c = tr.head[order[p]];
              if (q.tag >= 0 && c.tag >= 0 && te.adduct(q) != tr.adduct(c)) continue;
              bool same_instr = q.tag >= 0 && c.tag >= 0 && te.instr(q) == tr.instr(c);
              if (instr_mode == 2 && q.tag >= 0 && c.tag >= 0 && !same_instr) continue;
              const float* qmz = merge_mode ? merged.mz.data() : te.mzs(q);
              const float* qit = merge_mode ? merged.it.data() : te.its(q);
              int qn = merge_mode ? int(merged.mz.size()) : q.n;
              float qent = merge_mode ? merged.ent : te.ent[si];
              float s = simmode
                  ? entropy_sim(qmz, qit, qn, qent,
                                tr.mzs(c), tr.its(c), c.n, tr.ent[order[p]], tol)
                  : cosine(qmz, qit, qn,
                           tr.mzs(c), tr.its(c), c.n, tol);
              if (instr_mode == 1 && same_instr) s += instr_bonus;
              float dppm = (c.precursor - q.precursor) / q.precursor * 1e6f;
              float z = dppm / sigma;
              s -= alpha * z * z;
              auto it = best.find(c.mol);
              if (it == best.end()) best.emplace(c.mol, s);
              else it->second = std::max(it->second, s);
            }
          }
          if (pass == 0) tight_count[idx] = int(best.size());
          ppm_used *= 3.0f;
        }

        std::vector<std::pair<float, int> > rank;
        rank.reserve(best.size());
        for (const auto& kv : best) rank.emplace_back(kv.second, kv.first);
        std::sort(rank.begin(), rank.end(),
                  [](const std::pair<float, int>& a, const std::pair<float, int>& b) {
                    return a.first > b.first;
                  });

        std::string out;
        int placed = 0;
        for (size_t r = 0; r < rank.size() && placed < topk; ++r) {
          const std::string& smi = tr.smiles[rank[r].second];
          if (smi.empty()) continue;          // a molecule with no structure
          if (!out.empty()) out += ';';
          out += smi;
          ++placed;
          // On a holdout the query carries its own answer, so note where it
          // landed. The run then scores itself and no submission is spent
          // finding out whether a change helped.
          if (hit_rank[idx] < 0 && tr.name[rank[r].second] == te.name[mol_ids[idx]])
            hit_rank[idx] = placed;
        }
        answer[idx] = out;
      }
    });
  }
  for (std::thread& th : pool) th.join();

  std::string path = dir + "/submission.csv";
  std::ofstream out(path);
  out << "molecule_id,smiles\n";
  for (size_t i = 0; i < mol_ids.size(); ++i)
    out << te.name[mol_ids[i]] << ',' << answer[i] << '\n';
  std::printf("wrote %s for %zu molecules\n", path.c_str(), mol_ids.size());

  std::ofstream tc(dir + "/tight_count.csv");
  tc << "molecule_id,tight_count\n";
  for (size_t i = 0; i < mol_ids.size(); ++i)
    tc << te.name[mol_ids[i]] << ',' << tight_count[i] << '\n';

  int found = 0;
  for (int r : hit_rank) found += (r > 0);
  if (found) {
    double mrr = 0.0;
    int t1 = 0, t5 = 0;
    for (int r : hit_rank) {
      if (r <= 0) continue;
      mrr += 1.0 / r;
      t1 += (r == 1);
      t5 += (r <= 5);
    }
    size_t n = mol_ids.size();
    std::printf("top1 %.1f%%  top5 %.1f%%  top%d %.1f%%  MRR %.3f  (n=%zu)\n",
                100.0 * t1 / n, 100.0 * t5 / n, topk, 100.0 * found / n,
                mrr / n, n);
  }
  return 0;
}
"""


In [ ]:
EXPORT_SOURCE = r'''r"""Turn the parquet into something C++ can mmap and scan.

Two and a half million spectra will be compared against four hundred, which is
a billion peak-list comparisons if done naively. Parquet and pandas are the
wrong shape for that: the work is a tight loop over float arrays, so the data
goes out as flat binary and the loop goes in C++.

Layout, all little-endian:
  head.bin   one record per spectrum, fixed width
  peaks.bin  the m/z and intensity pairs, concatenated, pointed into by head
  smiles.txt one line per molecule, indexed by head.mol
"""
from __future__ import annotations

import struct, sys
from pathlib import Path

import numpy as np
import pyarrow.parquet as pq

ROOT = Path(sys.argv[1] if len(sys.argv) > 1 else r"C:\prog\casmi")

# Dropping peaks below this fraction of a spectrum's own base peak.
#
# ms2_mzs/ms2_normalized_intensities are not clean centroided peak lists: most
# spectra have a few dozen peaks (median 42), but a minority run into the tens
# of thousands (max measured 73,318), almost all of it near-zero-intensity
# filler. A cosine or entropy similarity computed over that is mostly scoring
# incidental overlap between two noise floors rather than real fragments,
# which dilutes genuine matches once a few hundred candidates share a mass
# window. Measured on a 400-molecule holdout: unfiltered MRR 0.43, with this
# floor 0.81 - and the gain is flat across a 20x range of the threshold
# (0.0005 to 0.01 all read 0.79-0.81), so the exact value is not load-bearing.
PEAK_FLOOR = 0.002

ADDUCTS = ["[M+H]+", "[M-H]-", "[M+Na]+", "[M+NH4]+", "[M+K]+", "[M+Cl]-",
           "[M+CH2O2-H]-", "[M+H-H2O]+", "[2M+H]+", "[2M+Na]+"]
ADDUCT_ID = {a: i for i, a in enumerate(ADDUCTS)}

INSTR = ["timsTOF", "Orbitrap", "QTOF", "LC-ESI-QTOF", "ESI-QFT"]
INSTR_ID = {a: i for i, a in enumerate(INSTR)}


def export(src: Path, out: str, with_labels: bool) -> None:
    cols = ["precursor_mz", "adduct", "instrument_type",
            "ms2_mzs", "ms2_normalized_intensities"]
    cols += ["inchikey14", "normalized_smiles"] if with_labels else ["molecule_id"]

    f = pq.ParquetFile(src)
    head = open(ROOT / f"{out}_head.bin", "wb")
    peaks = open(ROOT / f"{out}_peaks.bin", "wb")
    mols: dict[str, int] = {}
    names: list[str] = []
    smiles: list[str] = []
    off = 0
    n = 0

    for batch in f.iter_batches(batch_size=100000, columns=cols):
        d = batch.to_pandas()
        for r in d.itertuples(index=False):
            mz = np.asarray(r.ms2_mzs, dtype=np.float32)
            it = np.asarray(r.ms2_normalized_intensities, dtype=np.float32)
            k = min(len(mz), len(it))
            if k == 0:
                continue
            mz, it = mz[:k], it[:k]
            if k > 0 and it.max() > 0:
                keep = it >= PEAK_FLOOR * it.max()
                mz, it = mz[keep], it[keep]
            k = len(mz)
            if k == 0:
                continue
            o = np.argsort(mz)            # the comparison walks both lists in order
            peaks.write(mz[o].tobytes())
            peaks.write(it[o].tobytes())

            if with_labels:
                name = r.inchikey14 if isinstance(r.inchikey14, str) else ""
                smi = r.normalized_smiles if isinstance(r.normalized_smiles, str) else ""
            else:
                name, smi = r.molecule_id, ""
            if name not in mols:
                mols[name] = len(names)
                names.append(name)
                smiles.append(smi)
            elif smi and not smiles[mols[name]]:
                smiles[mols[name]] = smi

            head.write(struct.pack(
                "<fiiii",
                float(r.precursor_mz) if r.precursor_mz == r.precursor_mz else 0.0,
                mols[name], off, k,
                ADDUCT_ID.get(r.adduct, -1) * 16 + INSTR_ID.get(r.instrument_type, -1)))
            off += k
            n += 1
        print(f"  {n} spectra", end="\r", flush=True)

    head.close()
    peaks.close()
    (ROOT / f"{out}_mols.txt").write_text(
        "\n".join(f"{a}\t{b}" for a, b in zip(names, smiles)), encoding="utf-8")
    print(f"\n{out}: {n} spectra, {len(names)} molecules, {off} peaks")


export(ROOT / "train.parquet", "train", True)
export(ROOT / "test.parquet", "test", False)
'''


In [ ]:
# Spectral library search, compiled here and run over the competition data.
#
# The task is a search, not a model: every one of the 400 test molecules has
# its precursor mass somewhere in the 2.5 million training spectra, and the
# work is picking the right one out of the others that share that mass. Which
# makes the whole thing a tight loop over 400 million peaks, so it is C++.
#
# What was measured on a held-out 400 molecules shaped like the test set:
#   10 ppm mass gate ..... MRR 0.084   (median molecule competes with 105)
#   0.05 ppm gate ........ MRR 0.474
#   plus staged widening . MRR 0.481,  top25 94%
# The gate is the whole story; the cosine barely changed.
import os, subprocess, sys, textwrap, time
from pathlib import Path

t0 = time.time()
WORK = Path("/kaggle/working")
(WORK / "match.cpp").write_text(CPP_SOURCE, encoding="utf-8")
(WORK / "export_bin.py").write_text(EXPORT_SOURCE, encoding="utf-8")

r = subprocess.run(["g++", "-O3", "-std=c++17", "-pthread", "-w",
                    "-o", str(WORK / "match"), str(WORK / "match.cpp")],
                   capture_output=True, text=True)
print("compile rc:", r.returncode, flush=True)
if r.returncode != 0:
    print(r.stderr[-3000:])
    raise SystemExit(1)
print("compiled in %.0fs" % (time.time() - t0), flush=True)


In [ ]:
# Flatten the parquet into the binary the matcher scans.
#
# The competition directory is found rather than spelled out: the first attempt
# guessed the name from the URL slug and the run died having already compiled
# everything.
import glob, os
hits = glob.glob("/kaggle/input/**/train.parquet", recursive=True)
print("inputs:", os.listdir("/kaggle/input"), flush=True)
if not hits:
    raise SystemExit("no train.parquet under /kaggle/input")
DATA = Path(hits[0]).parent
print("data:", DATA, flush=True)

for name in ("train.parquet", "test.parquet"):
    dst = WORK / name
    if not dst.exists():
        os.symlink(DATA / name, dst)

r = subprocess.run([sys.executable, str(WORK / "export_bin.py"), str(WORK)],
                   capture_output=True, text=True)
print(r.stdout[-1500:], flush=True)
if r.returncode != 0:
    print(r.stderr[-3000:])
    raise SystemExit(1)
print("exported in %.0fs" % (time.time() - t0), flush=True)


In [ ]:
# The mass gate, and why it is this tight.
#
# train.parquet and test.parquet both carry precursor_mz rounded to exactly
# four decimals (measured: 99.9% and 100% of rows), and for all 400 test
# molecules the nearest training precursor is an exact bit match. The two files
# came off the same calculation, so the precursor is a key to look up rather
# than a measurement to tolerate: 0.05 ppm holds a median of 20 training
# spectra, 10 ppm holds 439 and all the extra ones are noise.
#
# That is worth stating because the opposite is the usual advice. The field
# runs 10 ppm windows, and a holdout with 3 ppm of simulated instrument error
# duly preferred wide gates. The error being simulated does not exist here.
#
# Measured on a 400-molecule holdout carved out of train, one spectrum moved to
# the queries and the rest of the molecule left in the library:
#   no mass prior, 15 ppm ..... MRR 0.052
#   0.05 ppm gate ............. MRR 0.468
#   plus Gaussian mass prior .. MRR 0.468
#   plus entropy + instrument bonus  MRR 0.43,  400/400 full lists
#   + a peak-intensity floor (0.2% of the spectrum's base peak, dropped
#     at export time) ............................ MRR 0.81
# ms2_mzs/ms2_normalized_intensities are not clean centroided peak lists:
# median 42 peaks/spectrum but a minority run into the tens of thousands
# (max measured 73,318), almost all near-zero filler. A cosine or entropy
# similarity over that is mostly scoring incidental overlap between two
# noise floors once a few hundred candidates share a mass window; a floor
# removes it. The gain is flat across a 20x range of the threshold
# (0.0005 to 0.01 all read 0.79-0.81), so the exact cutoff is not
# load-bearing, and once the noise is gone the mass prior and instrument
# bonus that mattered before (0.41 -> 0.43) shrink to within noise
# (0.803 without either, 0.81 with) - they were partly compensating for
# noise-corrupted scores.
r = subprocess.run([str(WORK / "match"), str(WORK), "0.05", "0.02", "25",
                    "train", "test", "3.0", "0.1", "1", "1", "0.25"],
                   capture_output=True, text=True)
print(r.stdout[-2000:], flush=True)
if r.returncode != 0:
    print(r.stderr[-2000:])
    raise SystemExit(1)

import pandas as pd
sub = pd.read_csv(WORK / "submission.csv")
n = sub.smiles.fillna("").str.count(";") + 1
print("direct-only: rows %d   full lists %d   fewest candidates %d   total %.0fs"
      % (len(sub), int((n == 25).sum()), int(n.min()), time.time() - t0))
(WORK / "direct_submission.csv").write_bytes((WORK / "submission.csv").read_bytes())
sub.head(3)


In [ ]:
ANALOG_SOURCE = r"""// Rank candidates that have no spectrum of their own, by borrowing spectral
// evidence from structurally similar candidates that do.
//
// Library search (match.cpp) can only ever answer a molecule whose structure
// already has a reference spectrum - about 16% of this competition's test set
// by the public baseline's own estimate. This tool handles the rest: for a
// candidate with no spectrum, it looks up the K structurally most similar
// candidates that DO have one (precomputed once by precompute_analogs.cpp,
// independent of any query - a candidate's chemical neighbours don't change
// per test molecule), scores each such donor's own reference spectrum against
// the query spectrum directly, and credits the candidate with
// donor_score * tanimoto(candidate, donor)^SIM_POWER, taking the best donor.
//
// Donors are NOT gated by mass to the query - that was tried and measured not
// to filter anything (library density across mass here is close to uniform:
// a +-200 Da window around a typical test precursor holds ~90% of the whole
// donor set). What IS gated by mass is which CANDIDATES are considered at
// all: a candidate only matters if its own (neutral, adduct-corrected) mass
// is close to what the query's precursor implies, exactly like direct search.
//
// Build:
//   g++ -O3 -std=c++17 -pthread -o analog_match analog_match.cpp
//
// Usage:
//   analog_match <dir> <ppm> <peak_tol> <topk> <qry_stem> [sim_power] [donor_k]

#include <algorithm>
#include <atomic>
#include <cmath>
#include <cstdint>
#include <cstdio>
#include <cstring>
#include <fstream>
#include <string>
#include <thread>
#include <unordered_map>
#include <vector>

namespace {

// Same order as ADDUCT_ID in the Python scripts - index into this table is
// h.tag / 16. The shift turns a precursor m/z back into the neutral,
// monoisotopic mass pool_head.bin is sorted by. [2M+*] adducts are dimers
// (2*M + adduct), which a single subtraction cannot undo exactly; they are
// left using the monomer's shift, which is wrong by roughly M itself for
// those rows - rare enough in practice that this is a known, accepted gap
// rather than a silent one.
constexpr float ADDUCT_SHIFT[10] = {
    1.007276f,    // [M+H]+
    -1.007276f,   // [M-H]-
    22.989218f,   // [M+Na]+
    18.033823f,   // [M+NH4]+
    38.963158f,   // [M+K]+
    34.969402f,   // [M+Cl]-
    44.998201f,   // [M+CH2O2-H]-  (+formic acid, -H)
    -17.002740f,  // [M+H-H2O]+
    1.007276f,    // [2M+H]+   (not dimer-corrected, see note above)
    22.989218f,   // [2M+Na]+  (not dimer-corrected, see note above)
};

struct Head {
  float precursor;
  int32_t mol, off, n, tag;
};

struct Store {
  std::vector<Head> head;
  std::vector<float> mz, inten;
  std::vector<std::string> name, smiles;
  const float* mzs(const Head& h) const { return mz.data() + h.off; }
  const float* its(const Head& h) const { return inten.data() + h.off; }
};

std::vector<char> slurp(const std::string& path) {
  std::ifstream f(path, std::ios::binary | std::ios::ate);
  if (!f) { std::fprintf(stderr, "cannot read %s\n", path.c_str()); std::exit(1); }
  std::streamsize n = f.tellg();
  f.seekg(0);
  std::vector<char> buf((size_t(n)));
  f.read(buf.data(), n);
  return buf;
}

// Loads a head.bin/peaks.bin/mols.txt triple with plain sqrt-cosine in mind -
// no entropy weighting here, since donors are only ever compared head to head
// against a query spectrum, no per-spectrum precomputed state required.
Store load(const std::string& dir, const std::string& stem) {
  Store s;
  std::vector<char> hb = slurp(dir + "/" + stem + "_head.bin");
  s.head.resize(hb.size() / sizeof(Head));
  std::memcpy(s.head.data(), hb.data(), s.head.size() * sizeof(Head));
  std::vector<char> pb = slurp(dir + "/" + stem + "_peaks.bin");
  size_t total = 0;
  for (const Head& h : s.head) total += size_t(h.n);
  s.mz.resize(total);
  s.inten.resize(total);
  const float* p = reinterpret_cast<const float*>(pb.data());
  size_t cur = 0;
  for (const Head& h : s.head) {
    std::memcpy(s.mz.data() + cur, p, size_t(h.n) * sizeof(float));
    p += h.n;
    std::memcpy(s.inten.data() + cur, p, size_t(h.n) * sizeof(float));
    p += h.n;
    cur += size_t(h.n);
  }
  std::ifstream f(dir + "/" + stem + "_mols.txt");
  std::string line;
  while (std::getline(f, line)) {
    if (!line.empty() && line.back() == '\r') line.pop_back();
    size_t t = line.find('\t');
    s.name.push_back(line.substr(0, t));
    s.smiles.push_back(t == std::string::npos ? "" : line.substr(t + 1));
  }
  std::printf("%s: %zu spectra, %zu molecules, %zu peaks\n",
              stem.c_str(), s.head.size(), s.name.size(), total);
  return s;
}

float cosine(const float* amz, const float* ait, int an,
             const float* bmz, const float* bit, int bn, float tol) {
  double dot = 0.0, na = 0.0, nb = 0.0;
  for (int i = 0; i < an; ++i) na += double(ait[i]);
  for (int j = 0; j < bn; ++j) nb += double(bit[j]);
  if (na <= 0.0 || nb <= 0.0) return 0.0f;
  int j = 0;
  for (int i = 0; i < an; ++i) {
    while (j < bn && bmz[j] < amz[i] - tol) ++j;
    int k = j;
    float best = 0.0f;
    while (k < bn && bmz[k] <= amz[i] + tol) { best = std::max(best, bit[k]); ++k; }
    if (best > 0.0f) dot += std::sqrt(double(ait[i])) * std::sqrt(double(best));
  }
  double norma = 0.0, normb = 0.0;
  for (int i = 0; i < an; ++i) norma += double(ait[i]);
  for (int j2 = 0; j2 < bn; ++j2) normb += double(bit[j2]);
  return float(dot / std::sqrt(norma * normb + 1e-12));
}

struct Pool {
  int n = 0;
  std::vector<float> mass;
  std::vector<int32_t> mol_id;        // original index into pool_mols.txt
  std::vector<int32_t> has_spectrum;
};

Pool load_pool(const std::string& dir) {
  Pool p;
  std::ifstream f(dir + "/pool_head.bin", std::ios::binary | std::ios::ate);
  if (!f) { std::fprintf(stderr, "cannot read pool_head.bin\n"); std::exit(1); }
  size_t bytes = f.tellg();
  f.seekg(0);
  p.n = int(bytes / 12);
  p.mass.resize(p.n); p.mol_id.resize(p.n); p.has_spectrum.resize(p.n);
  for (int i = 0; i < p.n; ++i) {
    f.read(reinterpret_cast<char*>(&p.mass[i]), 4);
    f.read(reinterpret_cast<char*>(&p.mol_id[i]), 4);
    f.read(reinterpret_cast<char*>(&p.has_spectrum[i]), 4);
  }
  return p;
}

std::vector<std::string> load_pool_mols(const std::string& dir, std::vector<std::string>* names_out) {
  std::vector<std::string> smiles;
  std::ifstream f(dir + "/pool_mols.txt");
  std::string line;
  while (std::getline(f, line)) {
    if (!line.empty() && line.back() == '\r') line.pop_back();
    size_t t = line.find('\t');
    names_out->push_back(line.substr(0, t));
    smiles.push_back(t == std::string::npos ? "" : line.substr(t + 1));
  }
  return smiles;
}

}  // namespace

int main(int argc, char** argv) {
  std::string dir = argc > 1 ? argv[1] : ".";
  float ppm = argc > 2 ? float(std::atof(argv[2])) : 50.0f;
  float tol = argc > 3 ? float(std::atof(argv[3])) : 0.02f;
  int topk = argc > 4 ? std::atoi(argv[4]) : 25;
  std::string qry_stem = argc > 5 ? argv[5] : "test";
  float sim_power = argc > 6 ? float(std::atof(argv[6])) : 3.0f;
  int donor_k = argc > 7 ? std::atoi(argv[7]) : 50;   // must match precompute_analogs's topk

  Pool pool = load_pool(dir);
  std::vector<std::string> pool_names;
  std::vector<std::string> pool_smiles = load_pool_mols(dir, &pool_names);
  std::printf("pool: %d candidates (%zu named)\n", pool.n, pool_names.size());

  Store donor = load(dir, "donor");      // one spectrum per train molecule
  Store te = load(dir, qry_stem);

  std::ifstream fd(dir + "/analog_donor.bin", std::ios::binary);
  std::ifstream fs(dir + "/analog_sim.bin", std::ios::binary);
  if (!fd || !fs) { std::fprintf(stderr, "missing analog_donor.bin/analog_sim.bin\n"); std::exit(1); }
  std::vector<int32_t> adonor(size_t(pool.n) * donor_k);
  std::vector<float> asim(size_t(pool.n) * donor_k);
  fd.read(reinterpret_cast<char*>(adonor.data()), adonor.size() * 4);
  fs.read(reinterpret_cast<char*>(asim.data()), asim.size() * 4);

  // Pool sorted by mass, for the candidate gate.
  std::vector<int> order(pool.n);
  for (int i = 0; i < pool.n; ++i) order[i] = i;
  std::sort(order.begin(), order.end(),
            [&](int a, int b) { return pool.mass[a] < pool.mass[b]; });
  std::vector<float> keys(pool.n);
  for (int i = 0; i < pool.n; ++i) keys[i] = pool.mass[order[i]];

  // donor Store's own "mol" field is the train molecule id, which equals
  // pool_mols.txt's row order for has_spectrum==1 entries (build_pool.py
  // appends train's structures first, unchanged) - so a donor mol_id doubles
  // directly as an index into donor.head if we also index donors by mol.
  std::vector<int> donor_row_of_mol(pool.n, -1);   // mol_id -> row in donor.head
  for (size_t i = 0; i < donor.head.size(); ++i) donor_row_of_mol[donor.head[i].mol] = int(i);

  std::unordered_map<int, std::vector<int> > by_mol;
  for (size_t i = 0; i < te.head.size(); ++i) by_mol[te.head[i].mol].push_back(int(i));
  std::vector<int> mol_ids;
  for (const auto& kv : by_mol) mol_ids.push_back(kv.first);
  std::sort(mol_ids.begin(), mol_ids.end());

  std::vector<std::string> answer(mol_ids.size());
  unsigned nthread = std::max(1u, std::thread::hardware_concurrency());
  std::vector<std::thread> pool_threads;
  std::atomic<size_t> next(0);

  for (unsigned t = 0; t < nthread; ++t) {
    pool_threads.emplace_back([&]() {
      std::unordered_map<int, float> donor_score_cache;   // donor mol_id -> score vs this query
      std::unordered_map<int, float> best;                // candidate mol_id -> best analog score
      for (;;) {
        size_t idx = next++;
        if (idx >= mol_ids.size()) break;
        best.clear();
        donor_score_cache.clear();

        for (int si : by_mol[mol_ids[idx]]) {
          const Head& q = te.head[si];
          int adduct_idx = q.tag < 0 ? -1 : q.tag / 16;
          float shift = (adduct_idx >= 0 && adduct_idx < 10) ? ADDUCT_SHIFT[adduct_idx] : 1.007276f;
          float neutral = q.precursor - shift;

          float ppm_used = ppm;
          for (int pass = 0; pass < 6; ++pass) {
            float window = neutral * ppm_used * 1e-6f;
            size_t lo = std::lower_bound(keys.begin(), keys.end(), neutral - window) - keys.begin();
            size_t hi = std::upper_bound(keys.begin(), keys.end(), neutral + window) - keys.begin();
            for (size_t p = lo; p < hi; ++p) {
              int row = order[p];
              if (pool.has_spectrum[row]) continue;   // direct search already covers these
              int cand_mol = pool.mol_id[row];
              const int32_t* dons = adonor.data() + size_t(cand_mol) * donor_k;
              const float* sims = asim.data() + size_t(cand_mol) * donor_k;
              float cand_score = 0.0f;
              for (int k = 0; k < donor_k && dons[k] >= 0; ++k) {
                int dmol = dons[k];
                auto itc = donor_score_cache.find(dmol);
                float ds;
                if (itc != donor_score_cache.end()) {
                  ds = itc->second;
                } else {
                  int drow = donor_row_of_mol[dmol];
                  ds = drow < 0 ? 0.0f
                                : cosine(te.mzs(q), te.its(q), q.n,
                                        donor.mzs(donor.head[drow]), donor.its(donor.head[drow]),
                                        donor.head[drow].n, tol);
                  donor_score_cache.emplace(dmol, ds);
                }
                float w = float(std::pow(double(sims[k]), double(sim_power)));
                cand_score = std::max(cand_score, ds * w);
              }
              auto it = best.find(cand_mol);
              if (it == best.end()) best.emplace(cand_mol, cand_score);
              else it->second = std::max(it->second, cand_score);
            }
            if (int(best.size()) >= topk) break;
            ppm_used *= 3.0f;
          }
        }

        // best is <cand_mol, score>; building the (score, cand_mol) rank pair
        // by range-constructing straight from best's own pair<const int,float>
        // silently swaps and type-converts both fields instead of erroring -
        // an int candidate id becomes a float "score" and the real score
        // truncates to an int id. Spell the field order out.
        std::vector<std::pair<float, int> > rank;
        rank.reserve(best.size());
        for (const auto& kv : best) rank.emplace_back(kv.second, kv.first);
        std::sort(rank.begin(), rank.end(),
                  [](const std::pair<float, int>& a, const std::pair<float, int>& b) {
                    return a.first > b.first;
                  });
        std::string out;
        int placed = 0;
        for (size_t r = 0; r < rank.size() && placed < topk; ++r) {
          const std::string& smi = pool_smiles[rank[r].second];
          if (smi.empty()) continue;
          if (!out.empty()) out += ';';
          out += smi;
          ++placed;
        }
        answer[idx] = out;
      }
    });
  }
  for (auto& th : pool_threads) th.join();

  std::string path = dir + "/analog_submission.csv";
  std::ofstream out(path);
  out << "molecule_id,smiles\n";
  for (size_t i = 0; i < mol_ids.size(); ++i)
    out << te.name[mol_ids[i]] << ',' << answer[i] << '\n';
  std::printf("wrote %s for %zu molecules\n", path.c_str(), mol_ids.size());
  return 0;
}
"""
MERGE_SOURCE = r'''r"""Combine the direct spectral search with the analog-propagation backfill.

match.cpp's staged widening fills exactly 25 slots almost regardless of
whether a real match exists nearby, by widening the mass window until it
does - so "already has 25 candidates" cannot tell a confident Class-1 hit
apart from 25 slots of padding found only because the window had to grow
huge. tight_count.csv is the separate signal: how many candidates the
TIGHTEST pass alone found, before any widening. Those are kept; the rest of
each molecule's 25 slots are filled from the analog-propagation list instead
of direct's own widened guesses, since analog candidates are chosen by
chemical similarity rather than "whatever was left in a much bigger window",
and only direct's widened leftovers pad out anything analog still could not
fill.

    python merge_submissions.py <direct.csv> <analog.csv> <tight_count.csv> <out.csv> [topk]
"""
from __future__ import annotations

import csv, sys
from pathlib import Path

direct_path, analog_path, tight_path, out_path = map(Path, sys.argv[1:5])
K = int(sys.argv[5]) if len(sys.argv) > 5 else 25

direct = {r[0]: r[1] for r in csv.reader(direct_path.open(encoding="utf-8"))}
analog = {r[0]: r[1] for r in csv.reader(analog_path.open(encoding="utf-8"))}
tight = {r[0]: int(r[1]) for r in csv.reader(tight_path.open(encoding="utf-8")) if r[0] != "molecule_id"}
direct.pop("molecule_id", None)
analog.pop("molecule_id", None)

rows = []
n_from_analog = []
for mol_id, dsmi in direct.items():
    d_list = [s for s in dsmi.split(";") if s]
    keep_n = min(tight.get(mol_id, K), len(d_list), K)
    kept = d_list[:keep_n]
    seen = set(kept)
    a_list = [s for s in analog.get(mol_id, "").split(";") if s and s not in seen]
    merged = kept + a_list
    if len(merged) < K:
        extra = [s for s in d_list[keep_n:] if s not in seen and s not in merged]
        merged += extra
    merged = merged[:K]
    n_from_analog.append(min(len(a_list), K - keep_n))
    rows.append((mol_id, ";".join(merged) if merged else "CCO"))

with out_path.open("w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["molecule_id", "smiles"])
    w.writerows(rows)
import numpy as np
n = np.array(n_from_analog)
print(f"wrote {len(rows)} rows; slots filled from analog propagation: "
      f"mean {n.mean():.1f}, molecules with >=1 analog slot: {(n>0).sum()}")
'''
(WORK / "analog_match.cpp").write_text(ANALOG_SOURCE, encoding="utf-8")
(WORK / "merge_submissions.py").write_text(MERGE_SOURCE, encoding="utf-8")

r = subprocess.run(["g++", "-O3", "-std=c++17", "-pthread", "-w",
                    "-o", str(WORK / "analog_match"), str(WORK / "analog_match.cpp")],
                   capture_output=True, text=True)
print("analog_match compile rc:", r.returncode, flush=True)
if r.returncode != 0:
    print(r.stderr[-3000:])
    raise SystemExit(1)


In [ ]:
# The analog-propagation pool, fingerprints and precomputed structural
# neighbours are built once on a dedicated machine, not recomputed on every
# submission: 729,389 candidates (train's own 275,810 plus everything new
# in COCONUT) against 275,810 donors is ~1.25e11 Tanimoto comparisons, a
# few minutes on 20 cores and not something to redo inside every kernel run.
# Shipped as a dataset instead:
#   pool_head.bin / pool_mols.txt   - every candidate structure, mass-sorted
#   donor_head.bin / donor_peaks.bin - one representative spectrum per
#                                      train molecule (dedup from 2.5M to
#                                      275,810; a +-200 Da mass-shift window
#                                      was measured to hold ~90% of the
#                                      whole donor set here, so donors are
#                                      not mass-gated at all, only candidates
#                                      are, against the query's own mass)
#   analog_donor.bin / analog_sim.bin - each spectrum-less candidate's 50
#                                        most structurally similar donors
POOL_DIR = next(p.parent for p in Path("/kaggle/input").rglob("pool_head.bin"))
print("pool dataset:", POOL_DIR, flush=True)
for name in ("pool_head.bin", "pool_mols.txt", "analog_donor.bin", "analog_sim.bin",
            "donor_head.bin", "donor_peaks.bin"):
    dst = WORK / name
    if not dst.exists():
        os.symlink(POOL_DIR / name, dst)

r = subprocess.run([str(WORK / "analog_match"), str(WORK), "50", "0.02", "25",
                    "test", "3.0", "50"], capture_output=True, text=True)
print(r.stdout[-2000:], flush=True)
if r.returncode != 0:
    print(r.stderr[-2000:])
    raise SystemExit(1)


In [ ]:
# match.exe fills exactly 25 slots almost regardless of whether a real
# match exists nearby, by widening the mass window until it does - so
# "already has 25" cannot tell a confident direct hit apart from 25 slots
# padded out by a much wider, weaker search. tight_count.csv (written by
# match.exe alongside submission.csv) is the number of candidates the
# TIGHTEST pass alone found; those are kept, and the remaining slots come
# from analog propagation instead of from match's own widened guesses.
r = subprocess.run([sys.executable, str(WORK / "merge_submissions.py"),
                    str(WORK / "direct_submission.csv"), str(WORK / "analog_submission.csv"),
                    str(WORK / "tight_count.csv"), str(WORK / "merged_submission.csv")],
                   capture_output=True, text=True)
print(r.stdout, flush=True)
if r.returncode != 0:
    print(r.stderr[-2000:])
    raise SystemExit(1)

import pandas as pd
# Take the row order from the sample submission rather than trusting ours.
sample = pd.read_csv(DATA / "sample_submission.csv")
sub = pd.read_csv(WORK / "merged_submission.csv").set_index("molecule_id")
missing = [m for m in sample.molecule_id if m not in sub.index]
if missing:
    print("MISSING", len(missing), missing[:5], flush=True)
sample["smiles"] = [sub.smiles.get(m, "CCO") for m in sample.molecule_id]
sample.to_csv(WORK / "submission.csv", index=False)
n = sample.smiles.fillna("").str.count(";") + 1
print("final: rows %d   full lists %d   fewest candidates %d   total %.0fs"
      % (len(sample), int((n == 25).sum()), int(n.min()), time.time() - t0))
sample.head(3)
